# #69 2016~2024 총세출 분모 민감도 분석

## tl;dr

- 지역통합 3개 회계 순계를 주 기준으로 4개 대안의 수준·순위·증감 방향을 비교한다.
- 지역통합 대안은 주 기준과 높은 순위상관을 보인다.
- 광역본청 대안은 범위 불일치 신호가 커 진단용으로만 사용한다.

## Context & Methods

### Key Assumptions

- 연도별 17개 지역 순위는 비율이 클수록 1위로 계산한다.
- 순위 강건성은 연도별 Spearman 상관과 절대 순위차로 평가한다.
- 시계열 강건성은 2017~2024년 전년 대비 증감 방향 일치율로 평가한다.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INPUT_PATH = ROOT / "data/processed/analysis/2016-2024_재정대응지수_총세출_민감도_패널.csv"
SUMMARY_PATH = ROOT / "data/processed/analysis/2016-2024_총세출_분모_민감도_요약.csv"
CANDIDATE_PATH = ROOT / "data/processed/analysis/2016-2024_총세출_분모_순위변동_후보.csv"
MAIN_VARIANT = "지역통합_3개회계_순계"
VALUE = "계획예산_총세출비율_pct"
KEY = ["지역", "연도"]

## Data

In [2]:
# 1. 입력과 765키 계약 확인
data = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")
assert len(data) == 765
assert data["분모대안"].nunique() == 5
assert data.duplicated(["분모대안", *KEY]).sum() == 0
assert np.isfinite(data[VALUE]).all()
assert data.groupby("분모대안").size().eq(153).all()
print("sha256:", hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest())
display(data.groupby("분모대안").size().rename("행").to_frame())

sha256: d79df13034bcaebf6484442e9a8a2a1ff6a289b0ff181e5320926740674c3f33


,행
분모대안,
광역본청_3개회계_순계,153
지역통합_3개회계_순계,153
지역통합_3개회계_총계,153
지역통합_4개회계_총계,153
지역통합_일반회계_순계,153


## Results

In [3]:
# 2. 주 기준 대비 순위와 비율 배율
ranked = data.copy()
ranked["순위"] = ranked.groupby(["분모대안", "연도"])[VALUE].rank(ascending=False, method="min")
main = ranked.loc[ranked["분모대안"].eq(MAIN_VARIANT), KEY + [VALUE, "순위"]].rename(
    columns={VALUE: "주기준_비율_pct", "순위": "주기준_순위"}
)
comparison = ranked.merge(main, on=KEY, how="left", validate="many_to_one")
comparison["순위차"] = comparison["순위"] - comparison["주기준_순위"]
comparison["절대순위차"] = comparison["순위차"].abs()
comparison["주기준대비_배율"] = comparison[VALUE] / comparison["주기준_비율_pct"]

yearly_correlation = (
    comparison.groupby(["분모대안", "연도"])
    .apply(
        lambda group: group[VALUE].corr(group["주기준_비율_pct"], method="spearman"),
        include_groups=False,
    )
    .rename("순위상관_rho")
    .reset_index()
)

In [4]:
# 3. 전년 대비 증감 방향 일치
comparison = comparison.sort_values(["분모대안", "지역", "연도"])
comparison["전년비_변화"] = comparison.groupby(["분모대안", "지역"])[VALUE].diff()
comparison["주기준_전년비_변화"] = comparison.groupby(["분모대안", "지역"])[
    "주기준_비율_pct"
].diff()
comparison["증감방향_일치"] = np.sign(comparison["전년비_변화"]).eq(
    np.sign(comparison["주기준_전년비_변화"])
)
direction = (
    comparison.loc[comparison["연도"].gt(2016)]
    .groupby("분모대안")["증감방향_일치"]
    .agg(방향일치건수="sum", 방향비교건수="size", 방향일치율="mean")
)

In [5]:
# 4. 대안별 민감도 요약
summary = (
    comparison.groupby("분모대안")
    .agg(
        비율중앙값_pct=(VALUE, "median"),
        주기준대비_배율중앙값=("주기준대비_배율", "median"),
        평균절대순위차=("절대순위차", "mean"),
        최대절대순위차=("절대순위차", "max"),
        순위5칸이상=("절대순위차", lambda values: values.ge(5).sum()),
        비율100pct초과=("100pct_초과", "sum"),
    )
    .join(
        yearly_correlation.groupby("분모대안")["순위상관_rho"].agg(
            순위상관_최솟값="min", 순위상관_중앙값="median"
        )
    )
    .join(direction)
    .reset_index()
)
summary["방향일치율_pct"] = summary["방향일치율"] * 100
display(summary.drop(columns="방향일치율"))

,분모대안,비율중앙값_pct,주기준대비_배율중앙값,평균절대순위차,최대절대순위차,순위5칸이상,비율100pct초과,순위상관_최솟값,순위상관_중앙값,방향일치건수,방향비교건수,방향일치율_pct
0,광역본청_3개회계_순계,56.004873,2.191320,5.751634,10.0,109,23,0.031863,0.149510,128,136,94.117647
1,지역통합_3개회계_순계,15.945094,1.000000,0.000000,0.0,0,0,1.000000,1.000000,136,136,100.000000
2,지역통합_3개회계_총계,11.898455,0.726612,1.006536,7.0,9,0,0.867647,0.921569,130,136,95.588235
3,지역통합_4개회계_총계,10.839991,0.673026,0.993464,7.0,9,0,0.823529,0.936275,130,136,95.588235
4,지역통합_일반회계_순계,19.198058,1.206696,0.601307,3.0,0,0,0.963235,0.980392,131,136,96.323529


In [6]:
# 5. 순위 변동 후보와 결과 저장
candidates = comparison.loc[
    comparison["분모대안"].ne(MAIN_VARIANT) & comparison["절대순위차"].ge(5),
    [
        "분모대안",
        "지역",
        "연도",
        "주기준_비율_pct",
        VALUE,
        "주기준_순위",
        "순위",
        "순위차",
        "절대순위차",
        "100pct_초과",
    ],
].sort_values(["절대순위차", "분모대안", "연도", "지역"], ascending=[False, True, True, True])

assert summary["분모대안"].nunique() == 5
assert len(candidates) == 127
assert candidates["분모대안"].eq("광역본청_3개회계_순계").sum() == 109
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")
candidates.to_csv(CANDIDATE_PATH, index=False, encoding="utf-8-sig")
display(candidates.head(10))
print("saved:", SUMMARY_PATH.relative_to(ROOT), "/", len(summary), "rows")
print("saved:", CANDIDATE_PATH.relative_to(ROOT), "/", len(candidates), "rows")

,분모대안,지역,연도,주기준_비율_pct,계획예산_총세출비율_pct,주기준_순위,순위,순위차,절대순위차,100pct_초과
28,광역본청_3개회계_순계,경북,2017,11.648334,73.861617,15.0,5.0,-10.0,10.0,False
18,광역본청_3개회계_순계,경남,2016,10.559678,52.728000,15.0,6.0,-9.0,9.0,False
27,광역본청_3개회계_순계,경북,2016,10.897275,59.090571,13.0,4.0,-9.0,9.0,False
128,광역본청_3개회계_순계,제주,2018,46.234733,46.234733,2.0,11.0,9.0,9.0,False
30,광역본청_3개회계_순계,경북,2019,12.700896,81.758535,14.0,5.0,-9.0,9.0,False
39,광역본청_3개회계_순계,광주,2019,20.419503,39.263646,1.0,10.0,9.0,9.0,False
112,광역본청_3개회계_순계,전남,2020,11.522674,77.874562,14.0,5.0,-9.0,9.0,False
50,광역본청_3개회계_순계,대구,2021,21.372877,42.268205,4.0,13.0,9.0,9.0,False
34,광역본청_3개회계_순계,경북,2023,14.093927,95.626784,15.0,6.0,-9.0,9.0,False
89,광역본청_3개회계_순계,세종,2024,18.359400,18.359400,7.0,16.0,9.0,9.0,False


saved: data/processed/analysis/2016-2024_총세출_분모_민감도_요약.csv / 5 rows
saved: data/processed/analysis/2016-2024_총세출_분모_순위변동_후보.csv / 127 rows


## Takeaways

- 세 지역통합 대안의 순위상관 중앙값은 0.92 이상이고 방향 일치율은 95% 이상이다.
- 일반회계 순계 대안이 주 기준과 가장 유사하다.
- 광역본청 대안은 순위와 수준이 크게 달라 주 지표에서 제외한다.